# 311 — Ablation over fifteen seeds (133J)

Full per-seed results for the eight-variant ablation, seeds **42–56**, at the
**published** co-training hyperparameters (`lambda_anchor = 1.0`, `lambda_cls = 0.2`).

| | |
|---|---|
| Notebook run | `GAAL_Ablation_133J.ipynb` |
| Driver | `_run_133J_seeds.py --seeds 42 43 44 45 46`, then `--seeds 47 48 49 50 51 52 53 54 55 56 --resume` |
| Regime | Fashion-MNIST, T-shirt/top · Pullover · Dress, pool 800:40:40, rho = 0.05 |
| Wall clock | 8 runs / 324.9 min for seeds 43–46; seed 42 run earlier; 20 runs / 862.1 min for seeds 47–56 |
| Artifacts | `saved/ablation_rows_133J_s<seed>_<backend>.csv` |

Every seed shares the same data split (`random_seed = 42` pins the pool, the test
set, the CV assignment and the train/val split), so the seeds form **paired blocks**
and variant-to-variant comparisons are like-for-like. The run seed drives only
method stochasticity — network init, batch order, GAN noise, acquisition RNG.

> **This notebook reports raw per-seed data.** Nothing here has been written to the
> paper. The aggregate at the end is provided for reference only.

## The eight variants

Rows form a **3×2 factorial** — selection rule × generative backbone — with the
unaugmented baseline below it and the proposed method above.

| Row | Mode | Backend | Classifier in loop | Backbone | Selection | Co-training |
|---|---|---|---|---|---|---|
| Baseline | `ab1` | — | No | none | none | No |
| Baseline 1 | `ab4` | bagan | No | BAGAN | none | No |
| Baseline 1a | `ab4` | bagan_gp | No | BAGAN-GP | none | No |
| Variant 1 | `ab8a` | bagan | Yes | BAGAN | margin | No |
| Variant 1a | `ab8a` | bagan_gp | Yes | BAGAN-GP | margin | No |
| Variant 2 | `ab8d` | bagan | Yes | BAGAN | confidence-gated | No |
| Variant 3 | `ab8d` | bagan_gp | Yes | BAGAN-GP | confidence-gated | No |
| Proposed | `ab8c` | bagan_gp | Yes | BAGAN-GP | confidence-gated | **Yes** |

`Baseline 1` / `Baseline 1a` generate a fully balanced synthetic set once, before
training, with **no selection at all** — they isolate what the generator alone buys.

**Precision is reported alongside the four headline metrics throughout.** It is not
in the paper's Table 2, but it is the variable that explains almost everything below:
the no-selection rows buy recall by sacrificing precision.

**Per-class FID (c0, c1, c2, minority mean) is reported as well, matching the layout of
Table 3 in the paper.** It is a property of the *generator*, not of the acquisition
method, so it is identical across every mode that shares a backbone and a seed:
`Baseline 1` = `Variant 1` = `Variant 2` on BAGAN, and `Baseline 1a` = `Variant 1a` =
`Variant 3` = `Proposed` on BAGAN-GP. `Baseline` has no generator and shows `n/a`.

**`Proposed`'s FID equals `Variant 3`'s in every seed here, even though co-training does
update the generator.** Section 3b below explains why: the retrain code computes a
real-anchored distance for both the co-trained generator and a frozen warm-up snapshot,
keeps whichever is closer to real data (logged as `[G-guard] ... (Ab-8b floor)`), and on
every one of the fifteen seeds the warm-up generator won — the co-trained weights are
never written to disk, so this notebook reports the FID the pipeline actually exports,
not a stand-in for what co-training alone would have produced.

In [ ]:
# --- Setup: load every per-seed CSV -----------------------------------------
import csv, json, statistics as st
from pathlib import Path

REPO = Path.cwd()
while not (REPO / "saved").exists() and REPO != REPO.parent:
    REPO = REPO.parent
SAVED, RUNS = REPO / "saved", REPO / "saved" / "nb_runs"
SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56]

ROWS = [
    (("bagan",    "ab1"),  "Baseline",    "none",     "none"),
    (("bagan",    "ab4"),  "Baseline 1",  "BAGAN",    "none"),
    (("bagan_gp", "ab4"),  "Baseline 1a", "BAGAN-GP", "none"),
    (("bagan",    "ab8a"), "Variant 1",   "BAGAN",    "margin"),
    (("bagan_gp", "ab8a"), "Variant 1a",  "BAGAN-GP", "margin"),
    (("bagan",    "ab8d"), "Variant 2",   "BAGAN",    "gate"),
    (("bagan_gp", "ab8d"), "Variant 3",   "BAGAN-GP", "gate"),
    (("bagan_gp", "ab8c"), "Proposed",    "BAGAN-GP", "gate+co-train"),
]

def load(seed):
    d = {}
    for be in ("bagan", "bagan_gp"):
        p = SAVED / f"ablation_rows_133J_s{seed}_{be}.csv"
        for r in csv.DictReader(p.open(encoding="utf-8")):
            d[(be, r["mode"])] = r
    return d

DATA = {s: load(s) for s in SEEDS}

def table(seed):
    """Table 2 layout: the four headline metrics, plus precision, per-class FID
    (c0/c1/c2/minority-mean, matching paper Table 3) and stop epoch."""
    d = DATA[seed]
    print(f"SEED {seed} — ablation metrics (held-out test set)")
    print(f"{'Row':<13}{'Backbone':<10}{'Selection':<15}"
          f"{'Recall':>8}{'G-mean':>9}{'F2':>8}{'Spec':>8}{'Prec':>8}"
          f"{'FID c0':>9}{'FID c1':>9}{'FID c2':>9}{'FID min':>9}{'Stop':>6}")
    print("-" * 130)
    best = {m: max(float(d[k][m]) for k, *_ in ROWS)
            for m in ("test_recall", "test_gmean", "test_f2", "test_specificity")}
    for key, label, bb, sel in ROWS:
        r = d[key]
        cells = ""
        for m in ("test_recall", "test_gmean", "test_f2", "test_specificity"):
            v = float(r[m])
            mark = "*" if abs(v - best[m]) < 5e-4 else " "
            cells += f"{v:>7.3f}{mark}" if m == "test_recall" else f"{v:>8.3f}{mark}"
        if r["fid_c0"] == "":
            fid_cells = f"{'n/a':>9}{'n/a':>9}{'n/a':>9}{'n/a':>9}"
        else:
            fid_cells = (f"{float(r['fid_c0']):>9.3f}{float(r['fid_c1']):>9.3f}"
                         f"{float(r['fid_c2']):>9.3f}{float(r['fid_minority_mean']):>9.3f}")
        print(f"{label:<13}{bb:<10}{sel:<15}{cells}"
              f"{float(r['test_precision']):>8.3f}"
              f"{fid_cells}"
              f"{int(float(r['avg_stopping_epoch'])):>6}")
    print("\n  * = best in column. FID is a property of (seed, backbone) as EXPORTED by the")
    print("  pipeline -- shared by every mode on that backbone. See the note above the first")
    print("  table for why Proposed and Variant 3 report identical FID despite co-training.")

def guard(seed):
    p = RUNS / f"GAAL_Ablation_133J_s{seed}_bagan_gp.ipynb"
    if not p.exists():
        return "run notebook unavailable"
    nb = json.loads(p.read_text(encoding="utf-8"))
    for c in nb["cells"]:
        for o in c.get("outputs", []):
            if o.get("output_type") == "stream":
                for l in "".join(o.get("text", "")).splitlines():
                    if "G-guard" in l:
                        return l.strip()
    return "no G-guard line found"

print(f"loaded seeds {SEEDS} from {SAVED}")

---

# Seed 42

| Row | Backbone | Selection | Recall | G-mean | $F2$ | Spec | Prec | FID c0 | FID c1 | FID c2 | FID min | Stop |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| Baseline | none | none | 0.795 | 0.782 | 0.805 | 0.901 | 0.855 | n/a | n/a | n/a | n/a | 140 |
| Baseline 1 | BAGAN | none | 0.845 | 0.837 | 0.801 | 0.919 | 0.692 | 165.944 | 147.603 | 245.969 | 196.786 | 91 |
| Baseline 1a | BAGAN-GP | none | **0.894** | **0.893** | 0.847 | **0.944** | 0.725 | 105.118 | 138.896 | 186.245 | 162.571 | 26 |
| Variant 1 | BAGAN | margin | 0.802 | 0.791 | 0.813 | 0.902 | 0.870 | 165.944 | 147.603 | 245.969 | 196.786 | 127 |
| Variant 1a | BAGAN-GP | margin | 0.863 | 0.858 | **0.848** | 0.930 | 0.801 | 105.118 | 138.896 | 186.245 | 162.571 | 116 |
| Variant 2 | BAGAN | gate | 0.856 | 0.851 | 0.842 | 0.929 | 0.799 | 165.944 | 147.603 | 245.969 | 196.786 | 111 |
| Variant 3 | BAGAN-GP | gate | 0.873 | 0.871 | 0.846 | 0.936 | 0.764 | 105.118 | 138.896 | 186.245 | 162.571 | 146 |
| Proposed | BAGAN-GP | gate+co-train | 0.873 | 0.870 | 0.847 | 0.933 | 0.767 | 105.118 | 138.896 | 186.245 | 162.571 | 259 |

**G-guard:** `[G-guard] warm-up G_ref closer to real (dist 1599.7 -> co-trained 6628.0) -> exporting warm-up G for FID (Ab-8b floor).`

In [ ]:
table(42)
print('\nG-guard:', guard(42))

### Insight — seed 42

**Baseline 1a leads three of four metrics** — recall 0.894, G-mean 0.893, specificity
0.944 — with no selection, no active learning and no co-training. It reaches its
stopping epoch in **26 epochs against Proposed's 259**, a 10× difference in training
budget for a better raw recall.

**But $F2$ is flat across all four BAGAN-GP rows**: 0.846–0.848, a spread of 0.002.
Precision is what separates them — 0.725 for Baseline 1a against 0.801 for Variant 1a.
Selection is buying back precision at the cost of recall, and $F2$ lands in the same
place either way.

**Co-training contributes nothing measurable.** Proposed and Variant 3 return identical
recall to four decimals (0.8727 both), differing by 0.001 on G-mean and 0.001 on $F2$ —
while Proposed trains 78% longer. The **G-guard fired**: the co-trained generator scored
6628.0 on the real-anchored distance against the warm-up generator's 1599.7 — 4.1× worse,
the largest gap of any seed — so the pipeline exported the warm-up generator instead.

*Read alone, this seed suggests dropping the acquisition machinery entirely. Seeds 43–46
show why that reading is premature.*

---

# Seed 43

| Row | Backbone | Selection | Recall | G-mean | $F2$ | Spec | Prec | FID c0 | FID c1 | FID c2 | FID min | Stop |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| Baseline | none | none | 0.807 | 0.795 | 0.813 | 0.900 | 0.846 | n/a | n/a | n/a | n/a | 96 |
| Baseline 1 | BAGAN | none | 0.828 | 0.816 | 0.795 | 0.912 | 0.716 | 156.318 | 130.420 | 159.927 | 145.173 | 60 |
| Baseline 1a | BAGAN-GP | none | 0.897 | 0.896 | 0.858 | 0.947 | 0.754 | 103.348 | 126.950 | 149.188 | 138.069 | 93 |
| Variant 1 | BAGAN | margin | 0.869 | 0.866 | 0.854 | 0.933 | 0.805 | 156.318 | 130.420 | 159.927 | 145.173 | 116 |
| Variant 1a | BAGAN-GP | margin | 0.856 | 0.851 | 0.840 | 0.929 | 0.796 | 103.348 | 126.950 | 149.188 | 138.069 | 111 |
| Variant 2 | BAGAN | gate | 0.844 | 0.839 | 0.839 | 0.918 | 0.824 | 156.318 | 130.420 | 159.927 | 145.173 | 100 |
| Variant 3 | BAGAN-GP | gate | **0.904** | **0.903** | **0.865** | **0.951** | 0.760 | 103.348 | 126.950 | 149.188 | 138.069 | 129 |
| Proposed | BAGAN-GP | gate+co-train | 0.849 | 0.845 | 0.838 | 0.923 | 0.797 | 103.348 | 126.950 | 149.188 | 138.069 | 247 |

**G-guard:** `[G-guard] warm-up G_ref closer to real (dist 1767.7 -> co-trained 4410.4) -> exporting warm-up G`

In [ ]:
table(43)
print('\nG-guard:', guard(43))

### Insight — seed 43

**The seed most favourable to the paper's thesis.** Variant 3 — the confidence gate on
the gradient-penalised backbone — is best on *all four* metrics: recall 0.904, G-mean
0.903, $F2$ 0.865, specificity 0.951. It beats Baseline 1a on every one.

**Co-training is actively harmful here.** Proposed drops to 0.849 recall against Variant
3's 0.904 — a **−0.055** cost from adding the third player, the largest single-seed
penalty in the whole study. Its $F2$ falls from 0.865 to 0.838.

The **G-guard fired**: the co-trained generator scored 4410.4 on the real-anchored
distance against the warm-up generator's 1767.7, so the pipeline discarded it and
exported the warm-up generator instead.

*If only this seed had been run, the paper's current claims would look well supported —
except for co-training, which even here makes things worse.*

---

# Seed 44

| Row | Backbone | Selection | Recall | G-mean | $F2$ | Spec | Prec | FID c0 | FID c1 | FID c2 | FID min | Stop |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| Baseline | none | none | 0.795 | 0.783 | 0.807 | 0.895 | 0.866 | n/a | n/a | n/a | n/a | 249 |
| Baseline 1 | BAGAN | none | 0.853 | 0.846 | **0.850** | 0.929 | 0.843 | 315.879 | 158.280 | 265.140 | 211.710 | 125 |
| Baseline 1a | BAGAN-GP | none | **0.891** | **0.890** | 0.837 | **0.942** | 0.704 | 89.950 | 130.652 | 160.392 | 145.522 | 43 |
| Variant 1 | BAGAN | margin | 0.807 | 0.797 | 0.812 | 0.906 | 0.837 | 315.879 | 158.280 | 265.140 | 211.710 | 138 |
| Variant 1a | BAGAN-GP | margin | 0.863 | 0.857 | 0.847 | 0.933 | 0.796 | 89.950 | 130.652 | 160.392 | 145.522 | 126 |
| Variant 2 | BAGAN | gate | 0.851 | 0.846 | 0.843 | 0.927 | 0.815 | 315.879 | 158.280 | 265.140 | 211.710 | 127 |
| Variant 3 | BAGAN-GP | gate | 0.849 | 0.844 | 0.834 | 0.925 | 0.786 | 89.950 | 130.652 | 160.392 | 145.522 | 134 |
| Proposed | BAGAN-GP | gate+co-train | 0.868 | 0.865 | **0.850** | 0.932 | 0.791 | 89.950 | 130.652 | 160.392 | 145.522 | 310 |

**G-guard:** `[G-guard] warm-up G_ref closer to real (dist 1647.6 -> co-trained 3864.5) -> exporting warm-up G`

In [ ]:
table(44)
print('\nG-guard:', guard(44))

### Insight — seed 44

**Recall and $F2$ point at different rows.** Baseline 1a again takes recall (0.891) and
specificity (0.942), but its $F2$ is only 0.837 — *below* Proposed's 0.850 and Baseline 1's
0.850. Precision explains it: 0.704 for Baseline 1a against 0.843 for Baseline 1.

**Co-training helps at this seed**: Proposed beats Variant 3 by +0.020 recall and +0.016
$F2$. That is the opposite sign to seed 43, and the two roughly cancel — which is the
central reason no co-training claim survives averaging.

Baseline 1 is unusually strong here (precision 0.843, $F2$ 0.850), the only seed where
unfiltered BAGAN augmentation does not damage precision. This is seed-level variance in
the warm-up generator, not a property of the method.

The **G-guard fired again** (1647.6 → 3864.5).

---

# Seed 45

| Row | Backbone | Selection | Recall | G-mean | $F2$ | Spec | Prec | FID c0 | FID c1 | FID c2 | FID min | Stop |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| Baseline | none | none | 0.794 | 0.780 | 0.801 | 0.897 | 0.844 | n/a | n/a | n/a | n/a | 122 |
| Baseline 1 | BAGAN | none | 0.869 | 0.865 | 0.779 | 0.923 | 0.614 | 209.245 | 154.130 | 213.094 | 183.612 | 118 |
| Baseline 1a | BAGAN-GP | none | **0.894** | **0.893** | 0.829 | **0.942** | 0.681 | 83.376 | 117.018 | 149.404 | 133.211 | 67 |
| Variant 1 | BAGAN | margin | 0.833 | 0.826 | 0.834 | 0.916 | 0.837 | 209.245 | 154.130 | 213.094 | 183.612 | 110 |
| Variant 1a | BAGAN-GP | margin | 0.826 | 0.818 | 0.828 | 0.913 | 0.836 | 83.376 | 117.018 | 149.404 | 133.211 | 114 |
| Variant 2 | BAGAN | gate | 0.800 | 0.790 | 0.807 | 0.897 | 0.839 | 209.245 | 154.130 | 213.094 | 183.612 | 145 |
| Variant 3 | BAGAN-GP | gate | 0.857 | 0.853 | **0.847** | 0.927 | 0.812 | 83.376 | 117.018 | 149.404 | 133.211 | 142 |
| Proposed | BAGAN-GP | gate+co-train | 0.818 | 0.811 | 0.817 | 0.908 | 0.814 | 83.376 | 117.018 | 149.404 | 133.211 | 277 |

**G-guard:** `[G-guard] warm-up G_ref closer to real (dist 1661.6 -> co-trained 1921.8) -> exporting warm-up G`

In [ ]:
table(45)
print('\nG-guard:', guard(45))

### Insight — seed 45

**The clearest demonstration that unfiltered augmentation is dangerous.** Baseline 1
posts recall 0.869 — above the unaugmented baseline's 0.794 — while its $F2$ *falls* to
0.779, the worst of any row. Precision collapses to **0.614** and accuracy to 0.884.
It is predicting minority classes indiscriminately.

Baseline 1a shows the same pathology in milder form: recall 0.894, $F2$ 0.829, precision
0.681.

**Variant 3 is the best row on $F2$ (0.847)** and holds precision at 0.812 — it recovers
most of the precision the no-selection rows throw away, while keeping recall at 0.857.

**Co-training costs again**: Proposed 0.818 recall against Variant 3's 0.857, a −0.039
penalty, echoing seed 43. The G-guard fired, though with the narrowest margin of the four
logged seeds (1661.6 → 1921.8).

---

# Seed 46

| Row | Backbone | Selection | Recall | G-mean | $F2$ | Spec | Prec | FID c0 | FID c1 | FID c2 | FID min | Stop |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| Baseline | none | none | 0.844 | 0.838 | 0.835 | 0.923 | 0.804 | n/a | n/a | n/a | n/a | 227 |
| Baseline 1 | BAGAN | none | 0.838 | 0.832 | 0.795 | 0.915 | 0.696 | 183.729 | 163.915 | 167.266 | 165.591 | 208 |
| Baseline 1a | BAGAN-GP | none | **0.889** | **0.886** | 0.759 | 0.930 | 0.576 | 108.804 | 137.480 | 161.348 | 149.414 | 198 |
| Variant 1 | BAGAN | margin | 0.858 | 0.854 | 0.853 | 0.925 | 0.836 | 183.729 | 163.915 | 167.266 | 165.591 | 118 |
| Variant 1a | BAGAN-GP | margin | 0.859 | 0.855 | 0.856 | 0.929 | 0.845 | 108.804 | 137.480 | 161.348 | 149.414 | 137 |
| Variant 2 | BAGAN | gate | 0.823 | 0.815 | 0.816 | 0.913 | 0.792 | 183.729 | 163.915 | 167.266 | 165.591 | 121 |
| Variant 3 | BAGAN-GP | gate | 0.798 | 0.787 | 0.799 | 0.901 | 0.815 | 108.804 | 137.480 | 161.348 | 149.414 | 126 |
| Proposed | BAGAN-GP | gate+co-train | 0.881 | 0.879 | **0.860** | **0.938** | 0.791 | 108.804 | 137.480 | 161.348 | 149.414 | 238 |

**G-guard:** `[G-guard] warm-up G_ref closer to real (dist 1664.7 -> co-trained 6461.9) -> exporting warm-up G`

In [ ]:
table(46)
print('\nG-guard:', guard(46))

### Insight — seed 46

**The decisive seed against no-selection.** Baseline 1a keeps its usual recall (0.889) but
its **$F2$ collapses to 0.759**, precision to **0.576** and accuracy to **0.853** — far below
the unaugmented baseline's 0.804 / 0.954. Adding 1216 unfiltered synthetic images made the
classifier substantially worse on the headline metric than doing nothing at all.

**Proposed is the best row here** — recall 0.881, $F2$ 0.860, specificity 0.938 — its
strongest showing of all fifteen seeds, beating Variant 3 by +0.083 recall.

**Variant 3 is the *worst* BAGAN-GP row** at 0.798 recall. The confidence gate and the
co-training loop swap places completely relative to seeds 43 and 45.

The **G-guard fired with the widest margin** of all: the co-trained generator scored 6461.9
against the warm-up generator's 1664.7, roughly 3.9× worse.

*Any claim resting on a single seed can be contradicted by picking a different one. This
seed and seed 43 are near mirror images.*

---

# Seed 47

| Row | Backbone | Selection | Recall | G-mean | $F2$ | Spec | Prec | FID c0 | FID c1 | FID c2 | FID min | Stop |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| Baseline | none | none | 0.796 | 0.782 | 0.810 | 0.896 | 0.887 | n/a | n/a | n/a | n/a | 100 |
| Baseline 1 | BAGAN | none | 0.787 | 0.769 | 0.794 | 0.894 | 0.850 | 163.654 | 132.088 | 152.208 | 142.148 | 239 |
| Baseline 1a | BAGAN-GP | none | **0.897** | **0.897** | 0.803 | **0.938** | 0.631 | 80.718 | 115.006 | 146.195 | 130.600 | 45 |
| Variant 1 | BAGAN | margin | 0.850 | 0.845 | 0.842 | 0.921 | 0.814 | 163.654 | 132.088 | 152.208 | 142.148 | 113 |
| Variant 1a | BAGAN-GP | margin | 0.874 | 0.871 | 0.850 | 0.937 | 0.775 | 80.718 | 115.006 | 146.195 | 130.600 | 142 |
| Variant 2 | BAGAN | gate | 0.798 | 0.789 | 0.802 | 0.895 | 0.820 | 163.654 | 132.088 | 152.208 | 142.148 | 134 |
| Variant 3 | BAGAN-GP | gate | 0.868 | 0.866 | **0.851** | 0.932 | 0.791 | 80.718 | 115.006 | 146.195 | 130.600 | 119 |
| Proposed | BAGAN-GP | gate+co-train | 0.830 | 0.824 | 0.826 | 0.911 | 0.808 | 80.718 | 115.006 | 146.195 | 130.600 | 253 |

**G-guard:** `[G-guard] warm-up G_ref closer to real (dist 2216.5 ->co-trained 4011.7) ->exporting warm-up G for FID (Ab-8b floor).`

In [ ]:
table(47)
print('\nG-guard:', guard(47))

### Insight — seed 47

**Baseline 1a sweeps recall, G-mean and specificity** (0.897 / 0.897 / 0.938) but its $F2$ is only 0.803 — precision has collapsed to 0.631, on par with the worst precision seen at seeds 45–46. **Variant 3 takes $F2$** (0.851), the confidence gate recovering most of the precision the unfiltered set throws away.

**Co-training costs again**: Proposed trails Variant 3 by -0.038 recall and -0.025 $F2$. The G-guard fired (warm-up 2216.5 vs co-trained 4011.7, 1.81×).

---

# Seed 48

| Row | Backbone | Selection | Recall | G-mean | $F2$ | Spec | Prec | FID c0 | FID c1 | FID c2 | FID min | Stop |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| Baseline | none | none | 0.826 | 0.818 | 0.826 | 0.919 | 0.827 | n/a | n/a | n/a | n/a | 187 |
| Baseline 1 | BAGAN | none | 0.808 | 0.798 | 0.778 | 0.902 | 0.707 | 178.470 | 132.041 | 206.913 | 169.477 | 74 |
| Baseline 1a | BAGAN-GP | none | **0.890** | **0.887** | 0.763 | 0.932 | 0.580 | 88.062 | 123.481 | 162.590 | 143.035 | 121 |
| Variant 1 | BAGAN | margin | 0.828 | 0.821 | 0.834 | 0.914 | 0.863 | 178.470 | 132.041 | 206.913 | 169.477 | 118 |
| Variant 1a | BAGAN-GP | margin | 0.828 | 0.822 | 0.815 | 0.915 | 0.780 | 88.062 | 123.481 | 162.590 | 143.035 | 122 |
| Variant 2 | BAGAN | gate | 0.822 | 0.815 | 0.813 | 0.912 | 0.780 | 178.470 | 132.041 | 206.913 | 169.477 | 134 |
| Variant 3 | BAGAN-GP | gate | 0.853 | 0.849 | **0.854** | 0.924 | 0.858 | 88.062 | 123.481 | 162.590 | 143.035 | 121 |
| Proposed | BAGAN-GP | gate+co-train | 0.867 | 0.864 | 0.844 | **0.934** | 0.773 | 88.062 | 123.481 | 162.590 | 143.035 | 247 |

**G-guard:** `[G-guard] warm-up G_ref closer to real (dist 1632.8 ->co-trained 5615.8) ->exporting warm-up G for FID (Ab-8b floor).`

In [ ]:
table(48)
print('\nG-guard:', guard(48))

### Insight — seed 48

**Another precision collapse on the no-selection row**: Baseline 1a's recall stays high (0.890) but $F2$ falls to 0.763 with precision at just 0.580. Unusually, **the confidence gate beats margin on precision here** (0.858 vs 0.780) as well as on $F2$ (0.854* vs 0.815).

Co-training is a wash: Proposed's recall is +0.014 over Variant 3, but $F2$ is -0.010 — a small gain traded for a small loss. G-guard: 1632.8 → 5615.8 (3.44×).

---

# Seed 49

| Row | Backbone | Selection | Recall | G-mean | $F2$ | Spec | Prec | FID c0 | FID c1 | FID c2 | FID min | Stop |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| Baseline | none | none | 0.795 | 0.781 | 0.806 | 0.895 | 0.866 | n/a | n/a | n/a | n/a | 127 |
| Baseline 1 | BAGAN | none | 0.787 | 0.766 | 0.793 | 0.893 | 0.834 | 275.890 | 272.554 | 175.099 | 223.827 | 110 |
| Baseline 1a | BAGAN-GP | none | **0.894** | **0.893** | **0.846** | **0.944** | 0.721 | 94.259 | 128.236 | 151.277 | 139.757 | 184 |
| Variant 1 | BAGAN | margin | 0.846 | 0.840 | 0.844 | 0.919 | 0.841 | 275.890 | 272.554 | 175.099 | 223.827 | 129 |
| Variant 1a | BAGAN-GP | margin | 0.839 | 0.833 | 0.839 | 0.919 | 0.839 | 94.259 | 128.236 | 151.277 | 139.757 | 122 |
| Variant 2 | BAGAN | gate | 0.843 | 0.838 | 0.833 | 0.923 | 0.808 | 275.890 | 272.554 | 175.099 | 223.827 | 138 |
| Variant 3 | BAGAN-GP | gate | 0.844 | 0.839 | 0.840 | 0.918 | 0.824 | 94.259 | 128.236 | 151.277 | 139.757 | 124 |
| Proposed | BAGAN-GP | gate+co-train | 0.837 | 0.831 | 0.830 | 0.914 | 0.806 | 94.259 | 128.236 | 151.277 | 139.757 | 280 |

**G-guard:** `[G-guard] warm-up G_ref closer to real (dist 1976.0 ->co-trained 4265.0) ->exporting warm-up G for FID (Ab-8b floor).`

In [ ]:
table(49)
print('\nG-guard:', guard(49))

### Insight — seed 49

**The one seed, of all fifteen, where the no-selection row wins outright.** Baseline 1a takes all four headline metrics — recall 0.894, G-mean 0.893, $F2$ 0.846, specificity 0.944 — with no acquisition machinery at all. Every other seed shows selection winning $F2$ or trading it off; this one shows neither.

Co-training is mildly negative: Proposed -0.008 recall, -0.010 $F2$ against Variant 3. G-guard: 1976.0 → 4265.0 (2.16×).

*A single counter-example like this is exactly why the aggregate, not any one seed, has to carry the paper's claim.*

---

# Seed 50

| Row | Backbone | Selection | Recall | G-mean | $F2$ | Spec | Prec | FID c0 | FID c1 | FID c2 | FID min | Stop |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| Baseline | none | none | 0.782 | 0.768 | 0.797 | 0.889 | 0.876 | n/a | n/a | n/a | n/a | 276 |
| Baseline 1 | BAGAN | none | 0.848 | 0.844 | 0.834 | 0.919 | 0.784 | 171.421 | 205.617 | 287.133 | 246.375 | 121 |
| Baseline 1a | BAGAN-GP | none | **0.913** | **0.913** | **0.859** | **0.953** | 0.723 | 79.911 | 124.973 | 142.354 | 133.664 | 71 |
| Variant 1 | BAGAN | margin | 0.858 | 0.856 | 0.814 | 0.922 | 0.702 | 171.421 | 205.617 | 287.133 | 246.375 | 129 |
| Variant 1a | BAGAN-GP | margin | 0.837 | 0.832 | 0.831 | 0.917 | 0.811 | 79.911 | 124.973 | 142.354 | 133.664 | 146 |
| Variant 2 | BAGAN | gate | 0.850 | 0.846 | 0.841 | 0.920 | 0.810 | 171.421 | 205.617 | 287.133 | 246.375 | 130 |
| Variant 3 | BAGAN-GP | gate | 0.818 | 0.809 | 0.816 | 0.911 | 0.808 | 79.911 | 124.973 | 142.354 | 133.664 | 129 |
| Proposed | BAGAN-GP | gate+co-train | 0.839 | 0.832 | 0.835 | 0.922 | 0.827 | 79.911 | 124.973 | 142.354 | 133.664 | 259 |

**G-guard:** `[G-guard] warm-up G_ref closer to real (dist 1750.9 ->co-trained 2984.3) ->exporting warm-up G for FID (Ab-8b floor).`

In [ ]:
table(50)
print('\nG-guard:', guard(50))

### Insight — seed 50

**Baseline 1a's strongest showing across all fifteen seeds**: recall 0.913, G-mean 0.913, $F2$ 0.859, specificity 0.953 — all four starred. Both selection rules under-perform it on $F2$ this time, and **Variant 3 is the weakest BAGAN-GP row of the seed** at 0.816, below even Variant 1a's 0.831.

**Co-training helps here**: Proposed beats Variant 3 by +0.021 recall and +0.019 $F2$ — one of only a few seeds where it does. The G-guard still fires, but with its narrowest margin of the 47–56 batch (warm-up 1750.9 → co-trained 2984.3, 1.70×), close to seed 45's low.

---

# Seed 51

| Row | Backbone | Selection | Recall | G-mean | $F2$ | Spec | Prec | FID c0 | FID c1 | FID c2 | FID min | Stop |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| Baseline | none | none | 0.826 | 0.818 | 0.827 | 0.909 | 0.835 | n/a | n/a | n/a | n/a | 136 |
| Baseline 1 | BAGAN | none | 0.825 | 0.818 | 0.802 | 0.911 | 0.740 | 168.829 | 162.858 | 167.846 | 165.352 | 41 |
| Baseline 1a | BAGAN-GP | none | **0.897** | **0.896** | 0.838 | **0.944** | 0.699 | 105.014 | 143.753 | 166.463 | 155.108 | 34 |
| Variant 1 | BAGAN | margin | 0.820 | 0.809 | 0.822 | 0.916 | 0.838 | 168.829 | 162.858 | 167.846 | 165.352 | 119 |
| Variant 1a | BAGAN-GP | margin | 0.831 | 0.824 | 0.826 | 0.914 | 0.810 | 105.014 | 143.753 | 166.463 | 155.108 | 116 |
| Variant 2 | BAGAN | gate | 0.832 | 0.824 | 0.829 | 0.915 | 0.825 | 168.829 | 162.858 | 167.846 | 165.352 | 116 |
| Variant 3 | BAGAN-GP | gate | 0.872 | 0.870 | **0.842** | 0.938 | 0.759 | 105.014 | 143.753 | 166.463 | 155.108 | 129 |
| Proposed | BAGAN-GP | gate+co-train | 0.862 | 0.858 | 0.825 | 0.932 | 0.723 | 105.014 | 143.753 | 166.463 | 155.108 | 265 |

**G-guard:** `[G-guard] warm-up G_ref closer to real (dist 1950.8 ->co-trained 7859.6) ->exporting warm-up G for FID (Ab-8b floor).`

In [ ]:
table(51)
print('\nG-guard:', guard(51))

### Insight — seed 51

Baseline 1a again leads recall, G-mean and specificity (0.897 / 0.896 / 0.944); Variant 3 takes $F2$ narrowly (0.842). Proposed trails Variant 3 by -0.010 recall and -0.017 $F2$ — the co-training penalty seen at seeds 43, 45, 47 and 54 recurring again.

The G-guard fired with one of the widest margins in the study: warm-up 1950.8 against co-trained 7859.6, a 4.03× gap.

---

# Seed 52

| Row | Backbone | Selection | Recall | G-mean | $F2$ | Spec | Prec | FID c0 | FID c1 | FID c2 | FID min | Stop |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| Baseline | none | none | 0.851 | 0.845 | 0.843 | 0.924 | 0.820 | n/a | n/a | n/a | n/a | 151 |
| Baseline 1 | BAGAN | none | 0.895 | 0.893 | 0.778 | 0.933 | 0.597 | 195.829 | 152.690 | 165.665 | 159.177 | 74 |
| Baseline 1a | BAGAN-GP | none | **0.895** | **0.894** | 0.833 | **0.943** | 0.687 | 103.569 | 123.328 | 149.959 | 136.643 | 42 |
| Variant 1 | BAGAN | margin | 0.863 | 0.860 | 0.851 | 0.933 | 0.806 | 195.829 | 152.690 | 165.665 | 159.177 | 116 |
| Variant 1a | BAGAN-GP | margin | 0.871 | 0.868 | **0.862** | 0.935 | 0.829 | 103.569 | 123.328 | 149.959 | 136.643 | 118 |
| Variant 2 | BAGAN | gate | 0.862 | 0.860 | 0.827 | 0.929 | 0.727 | 195.829 | 152.690 | 165.665 | 159.177 | 126 |
| Variant 3 | BAGAN-GP | gate | 0.868 | 0.865 | 0.849 | 0.932 | 0.789 | 103.569 | 123.328 | 149.959 | 136.643 | 137 |
| Proposed | BAGAN-GP | gate+co-train | 0.883 | 0.881 | 0.845 | 0.940 | 0.742 | 103.569 | 123.328 | 149.959 | 136.643 | 295 |

**G-guard:** `[G-guard] warm-up G_ref closer to real (dist 1915.6 ->co-trained 6712.4) ->exporting warm-up G for FID (Ab-8b floor).`

In [ ]:
table(52)
print('\nG-guard:', guard(52))

### Insight — seed 52

**The precision pathology shows up on the plain BAGAN backbone too**: Baseline 1 (no selection, no GP) matches Baseline 1a's recall (0.895 vs 0.895) but at precision 0.597 — this isn't a BAGAN-GP-only effect, unfiltered augmentation damages precision on either backbone.

**Margin edges out the gate on $F2$ here** (0.862 vs 0.849), the reverse of seeds 47–48. Co-training is essentially flat: Proposed +0.015 recall, -0.005 $F2$ against Variant 3. G-guard: 1915.6 → 6712.4 (3.50×).

---

# Seed 53

| Row | Backbone | Selection | Recall | G-mean | $F2$ | Spec | Prec | FID c0 | FID c1 | FID c2 | FID min | Stop |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| Baseline | none | none | 0.756 | 0.731 | 0.772 | 0.879 | 0.871 | n/a | n/a | n/a | n/a | 175 |
| Baseline 1 | BAGAN | none | 0.862 | 0.861 | 0.791 | 0.917 | 0.642 | 148.116 | 233.199 | 188.996 | 211.097 | 45 |
| Baseline 1a | BAGAN-GP | none | **0.892** | **0.890** | 0.819 | **0.939** | 0.665 | 90.069 | 120.429 | 149.664 | 135.046 | 117 |
| Variant 1 | BAGAN | margin | 0.733 | 0.714 | 0.747 | 0.863 | 0.817 | 148.116 | 233.199 | 188.996 | 211.097 | 132 |
| Variant 1a | BAGAN-GP | margin | 0.869 | 0.865 | **0.852** | 0.932 | 0.795 | 90.069 | 120.429 | 149.664 | 135.046 | 132 |
| Variant 2 | BAGAN | gate | 0.792 | 0.779 | 0.797 | 0.893 | 0.832 | 148.116 | 233.199 | 188.996 | 211.097 | 142 |
| Variant 3 | BAGAN-GP | gate | 0.844 | 0.839 | 0.838 | 0.924 | 0.813 | 90.069 | 120.429 | 149.664 | 135.046 | 113 |
| Proposed | BAGAN-GP | gate+co-train | 0.862 | 0.859 | 0.844 | 0.932 | 0.791 | 90.069 | 120.429 | 149.664 | 135.046 | 265 |

**G-guard:** `[G-guard] warm-up G_ref closer to real (dist 2056.9 ->co-trained 9025.8) ->exporting warm-up G for FID (Ab-8b floor).`

In [ ]:
table(53)
print('\nG-guard:', guard(53))

### Insight — seed 53

**Both the unaugmented baseline and margin-selected BAGAN collapse this seed**: the plain baseline's recall drops to 0.756, the lowest of any seed's Baseline row, and Variant 1 (BAGAN, margin) falls further to 0.733 — the worst BAGAN row measured across all fifteen seeds. Baseline 1a is unaffected, leading recall/G-mean/specificity as usual (0.892 / 0.890 / 0.939); Variant 1a (margin, BAGAN-GP) takes $F2$ (0.852) over Variant 3's 0.838.

Co-training helps modestly: Proposed +0.017 recall, +0.007 $F2$ over Variant 3. The G-guard fired at 2056.9 → 9025.8, a 4.39× gap — the widest margin seen up to this point in the seed range.

---

# Seed 54

| Row | Backbone | Selection | Recall | G-mean | $F2$ | Spec | Prec | FID c0 | FID c1 | FID c2 | FID min | Stop |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| Baseline | none | none | 0.795 | 0.782 | 0.806 | 0.898 | 0.860 | n/a | n/a | n/a | n/a | 237 |
| Baseline 1 | BAGAN | none | 0.840 | 0.838 | 0.691 | 0.905 | 0.541 | 153.122 | 142.613 | 153.834 | 148.223 | 72 |
| Baseline 1a | BAGAN-GP | none | **0.904** | **0.903** | 0.827 | **0.945** | 0.668 | 102.817 | 136.146 | 148.143 | 142.145 | 92 |
| Variant 1 | BAGAN | margin | 0.826 | 0.815 | 0.824 | 0.915 | 0.827 | 153.122 | 142.613 | 153.834 | 148.223 | 129 |
| Variant 1a | BAGAN-GP | margin | 0.851 | 0.845 | 0.842 | 0.924 | 0.816 | 102.817 | 136.146 | 148.143 | 142.145 | 130 |
| Variant 2 | BAGAN | gate | 0.837 | 0.831 | 0.830 | 0.917 | 0.808 | 153.122 | 142.613 | 153.834 | 148.223 | 137 |
| Variant 3 | BAGAN-GP | gate | 0.876 | 0.873 | **0.858** | 0.939 | 0.798 | 102.817 | 136.146 | 148.143 | 142.145 | 129 |
| Proposed | BAGAN-GP | gate+co-train | 0.823 | 0.816 | 0.816 | 0.910 | 0.792 | 102.817 | 136.146 | 148.143 | 142.145 | 241 |

**G-guard:** `[G-guard] warm-up G_ref closer to real (dist 2309.4 ->co-trained 4784.1) ->exporting warm-up G for FID (Ab-8b floor).`

In [ ]:
table(54)
print('\nG-guard:', guard(54))

### Insight — seed 54

**The sharpest precision collapse in the entire study**: Baseline 1 (BAGAN, no selection) posts $F2$ = 0.691 with precision 0.541 — worse than seed 45's Baseline-1a low (precision 0.681) and seed 46's Baseline-1a low (0.576) put together. Unfiltered augmentation is actively destructive here on the plain BAGAN backbone. Baseline 1a still leads recall/G-mean/specificity (0.904 / 0.903 / 0.945); Variant 3 takes $F2$ (0.858).

**Co-training costs heavily**: Proposed trails Variant 3 by -0.053 recall and -0.041 $F2$ — on par with seed 43's −0.055/−0.027 as the largest co-training penalties measured in the whole fifteen-seed study. G-guard: 2309.4 → 4784.1 (2.07×).

---

# Seed 55

| Row | Backbone | Selection | Recall | G-mean | $F2$ | Spec | Prec | FID c0 | FID c1 | FID c2 | FID min | Stop |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| Baseline | none | none | 0.856 | 0.851 | **0.842** | 0.929 | 0.795 | n/a | n/a | n/a | n/a | 138 |
| Baseline 1 | BAGAN | none | 0.851 | 0.849 | 0.806 | 0.921 | 0.691 | 181.619 | 164.194 | 163.200 | 163.697 | 35 |
| Baseline 1a | BAGAN-GP | none | **0.893** | **0.893** | 0.809 | **0.938** | 0.645 | 106.916 | 129.774 | 161.514 | 145.644 | 145 |
| Variant 1 | BAGAN | margin | 0.782 | 0.768 | 0.798 | 0.889 | 0.884 | 181.619 | 164.194 | 163.200 | 163.697 | 126 |
| Variant 1a | BAGAN-GP | margin | 0.818 | 0.810 | 0.818 | 0.908 | 0.819 | 106.916 | 129.774 | 161.514 | 145.644 | 145 |
| Variant 2 | BAGAN | gate | 0.837 | 0.834 | 0.812 | 0.917 | 0.733 | 181.619 | 164.194 | 163.200 | 163.697 | 130 |
| Variant 3 | BAGAN-GP | gate | 0.845 | 0.840 | 0.841 | 0.918 | 0.828 | 106.916 | 129.774 | 161.514 | 145.644 | 129 |
| Proposed | BAGAN-GP | gate+co-train | 0.870 | 0.867 | 0.838 | 0.930 | 0.744 | 106.916 | 129.774 | 161.514 | 145.644 | 226 |

**G-guard:** `[G-guard] warm-up G_ref closer to real (dist 1580.7 ->co-trained 7222.9) ->exporting warm-up G for FID (Ab-8b floor).`

In [ ]:
table(55)
print('\nG-guard:', guard(55))

### Insight — seed 55

**The unaugmented baseline beats every augmented row on $F2$.** At 0.842, the plain classifier with no generator, no selection and no co-training posts the highest $F2$ of all eight rows this seed — the only one of fifteen where that happens. Every mechanism in the pipeline underperforms doing nothing here.

**The G-guard fires with the widest margin of the entire study**: warm-up 1580.7 against co-trained 7222.9, a 4.57× gap. Proposed and Variant 3 split: Proposed leads recall by +0.025 but trails on $F2$ by -0.003.

*If the paper's evidence rested on this seed alone, none of its claims would survive.*

---

# Seed 56

| Row | Backbone | Selection | Recall | G-mean | $F2$ | Spec | Prec | FID c0 | FID c1 | FID c2 | FID min | Stop |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| Baseline | none | none | 0.802 | 0.791 | 0.814 | 0.899 | 0.874 | n/a | n/a | n/a | n/a | 108 |
| Baseline 1 | BAGAN | none | 0.850 | 0.842 | 0.817 | 0.923 | 0.735 | 175.564 | 172.587 | 171.183 | 171.885 | 179 |
| Baseline 1a | BAGAN-GP | none | **0.906** | **0.905** | 0.833 | **0.946** | 0.676 | 98.351 | 131.467 | 157.068 | 144.267 | 146 |
| Variant 1 | BAGAN | margin | 0.781 | 0.768 | 0.794 | 0.891 | 0.858 | 175.564 | 172.587 | 171.183 | 171.885 | 127 |
| Variant 1a | BAGAN-GP | margin | 0.845 | 0.839 | **0.842** | 0.922 | 0.833 | 98.351 | 131.467 | 157.068 | 144.267 | 130 |
| Variant 2 | BAGAN | gate | 0.801 | 0.791 | 0.811 | 0.901 | 0.857 | 175.564 | 172.587 | 171.183 | 171.885 | 130 |
| Variant 3 | BAGAN-GP | gate | 0.861 | 0.858 | **0.841** | 0.931 | 0.782 | 98.351 | 131.467 | 157.068 | 144.267 | 122 |
| Proposed | BAGAN-GP | gate+co-train | 0.861 | 0.857 | 0.841 | 0.931 | 0.776 | 98.351 | 131.467 | 157.068 | 144.267 | 247 |

**G-guard:** `[G-guard] warm-up G_ref closer to real (dist 1928.3 ->co-trained 8267.6) ->exporting warm-up G for FID (Ab-8b floor).`

In [ ]:
table(56)
print('\nG-guard:', guard(56))

### Insight — seed 56

Baseline 1a sweeps recall/G-mean/specificity again (0.906 / 0.905 / 0.946); Variant 1a (margin) and Variant 3 (gate) are essentially tied on $F2$ (0.842 vs 0.841).

**Co-training changes nothing, echoing seed 42**: Proposed and Variant 3 differ by -0.000 recall and -0.001 $F2$ — both effectively zero. The G-guard still fires (1928.3 → 8267.6, 4.29×), so this is the safeguard holding steady, not co-training doing nothing on its own.

*Fifteen seeds in, the pattern from seed 42 has repeated at the far end of the range: sometimes co-training's net effect is simply nil.*

---

# Aggregate over the fifteen seeds

Provided for reference. The per-seed tables above are the primary record.

In [ ]:
# --- Aggregate: mean +/- sd across the fifteen seeds ---------------------------
METRICS = [("test_recall","Recall"),("test_gmean","G-mean"),
           ("test_f2","F2"),("test_specificity","Spec"),("test_precision","Prec")]
print(f"{'Row':<13}" + "".join(f"{n:>17}" for _, n in METRICS))
print("-" * 98)
FID_KEYS = [("fid_c0","c0"), ("fid_c1","c1"), ("fid_c2","c2"), ("fid_minority_mean","min")]
for key, label, bb, sel in ROWS:
    cells = ""
    for m, _ in METRICS:
        col = [float(DATA[s][key][m]) for s in SEEDS]
        cells += f"{st.mean(col):.3f}+/-{st.stdev(col):.3f}".rjust(17)
    print(f"{label:<13}{cells}")
    fid_parts = []
    for fkey, fname in FID_KEYS:
        col = [float(DATA[s][key][fkey]) for s in SEEDS if DATA[s][key][fkey]]
        fid_parts.append(f"{fname}={st.mean(col):.2f}+/-{st.stdev(col):.2f}" if col else f"{fname}=n/a")
    print("    FID  " + "  ".join(fid_parts))

print("\n\nPer-seed F2, the four BAGAN-GP rows:")
print(f"{'seed':<6}{'Baseline 1a':>13}{'Variant 1a':>13}{'Variant 3':>12}{'Proposed':>11}")
for s in SEEDS:
    g = lambda k: float(DATA[s][k]["test_f2"])
    print(f"{s:<6}{g(('bagan_gp','ab4')):>13.3f}{g(('bagan_gp','ab8a')):>13.3f}"
          f"{g(('bagan_gp','ab8d')):>12.3f}{g(('bagan_gp','ab8c')):>11.3f}")

print("\n\nCo-training effect (Proposed - Variant 3), per seed:")
for s in SEEDS:
    dr = float(DATA[s][("bagan_gp","ab8c")]["test_recall"]) - float(DATA[s][("bagan_gp","ab8d")]["test_recall"])
    df = float(DATA[s][("bagan_gp","ab8c")]["test_f2"])     - float(DATA[s][("bagan_gp","ab8d")]["test_f2"])
    print(f"  seed {s}:  recall {dr:+.3f}   F2 {df:+.3f}")

print("\n\nProposed vs Variant 3, minority-mean FID, per seed (checking G-guard consistency):")
for s in SEEDS:
    fp = DATA[s][("bagan_gp","ab8c")]["fid_minority_mean"]
    fv = DATA[s][("bagan_gp","ab8d")]["fid_minority_mean"]
    same = "identical (G-guard reverted to warm-up G)" if fp == fv else f"DIFFERENT: Proposed={fp} Variant3={fv}"
    print(f"  seed {s}:  {same}")

## Aggregate table (mean ± sd, fifteen seeds)

| Row | Precision | Recall | G-mean | **$F2$** | Spec | FID c0 | FID c1 | FID c2 | FID min |
|---|---|---|---|---|---|---|---|---|---|
| Baseline | 0.848 ± 0.028 | 0.808 ± 0.027 | 0.796 ± 0.032 | 0.814 ± 0.019 | 0.904 ± 0.014 | n/a | n/a | n/a | n/a |
| Baseline 1 | 0.712 ± 0.091 | 0.839 ± 0.029 | 0.832 ± 0.034 | 0.794 ± 0.035 | 0.914 ± 0.012 | 189.58±46.59 | 168.32±39.87 | 192.23±42.67 | 180.28±31.11 |
| Baseline 1a | **0.676** ± 0.051 | **0.896 ± 0.007** | **0.895** ± 0.007 | 0.824 ± 0.030 | **0.942** ± 0.006 | **96.02±10.00** | **128.51±8.20** | **156.12±10.95** | **142.31±8.68** |
| Variant 1 | 0.829 ± 0.042 | 0.824 ± 0.038 | 0.815 ± 0.042 | 0.822 ± 0.028 | 0.911 ± 0.019 | 189.58±46.59 | 168.32±39.87 | 192.23±42.67 | 180.28±31.11 |
| Variant 1a | 0.812 ± 0.022 | 0.849 ± 0.018 | 0.843 ± 0.019 | 0.840 ± 0.014 | 0.924 ± 0.009 | **96.02±10.00** | **128.51±8.20** | **156.12±10.95** | **142.31±8.68** |
| Variant 2 | 0.805 ± 0.036 | 0.830 ± 0.023 | 0.823 ± 0.026 | 0.823 ± 0.016 | 0.914 ± 0.012 | 189.58±46.59 | 168.32±39.87 | 192.23±42.67 | 180.28±31.11 |
| **Variant 3** | 0.799 ± 0.028 | 0.855 ± 0.025 | 0.851 ± 0.028 | **0.841 ± 0.016** | 0.927 ± 0.012 | **96.02±10.00** | **128.51±8.20** | **156.12±10.95** | **142.31±8.68** |
| Proposed | 0.783 ± 0.029 | 0.855 ± 0.021 | 0.851 ± 0.023 | 0.837 ± 0.012 | 0.926 ± 0.011 | **96.02±10.00** | **128.51±8.20** | **156.12±10.95** | **142.31±8.68** |

At n=5 the confidence-gate row (Variant 3) trailed margin (Variant 1a) on $F2$ (0.838 vs 0.844). At n=15 that
has reversed by a razor-thin margin — Variant 3 0.8414 vs Variant 1a 0.8398 —
and the reversal itself is the evidence: a real effect would not flip which rule wins between the two seed
batches. See Insight 5.

## Insights

### 1. Recall and $F2$ crown different winners, and precision is why

Baseline 1a wins **recall** decisively — 0.896 with a standard deviation of just
**0.007**, the tightest of any row. But on **$F2$**, the paper's headline metric, it drops to
0.824 with a standard deviation of **0.030**, more than four times wider.

The mechanism is precision: **0.676 ± 0.051** for Baseline 1a against
0.848 for the unaugmented baseline. Pouring in a fully balanced, unfiltered synthetic set pushes the
classifier toward predicting minority classes indiscriminately. Recall rises, precision falls further, and $F2$ —
which weights recall four times as heavily as precision but does not ignore it — ends up worse and far less
predictable.

**This vindicates the choice of $F2$.** Had the paper reported recall alone, the honest conclusion would be that
the entire acquisition pipeline is unnecessary. $F2$ is the metric that exposes the trade.

### 2. Selection's real contribution is precision and stability — now statistically significant

On the BAGAN-GP backbone, moving from no selection to any selection rule:

| | Precision | Recall | $F2$ | $F2$ sd |
|---|---|---|---|---|
| Baseline 1a (none) | 0.676 | 0.896 | 0.824 | 0.030 |
| Variant 1a (margin) | 0.812 | 0.849 | 0.840 | **0.014** |
| Variant 3 (gate) | 0.799 | 0.855 | 0.841 | 0.016 |

Selection trades roughly **0.041–0.048 recall for
0.123–0.136 precision**, and cuts $F2$ variance by
1.84–2.17×.

**With n=15, this is no longer just descriptive.** The precision gain over no-selection is significant for both
rules — margin: $p$ = 6.1e-05; gate: $p$ = 6.1e-05 (Wilcoxon signed-rank, both at the smallest
$p$-value obtainable at this $n$, meaning every one of the 15 seeds moved the same direction). The $F2$ gain itself is
more equivocal: significant for the confidence gate ($p$ = 0.022) but not for margin ($p$ = 0.107) —
a genuine asymmetry the 5-seed data could not have resolved.

### 3. The backbone remains the largest effect, and the only one significant at every level

BAGAN → BAGAN-GP, holding the selection rule fixed (mean recall, Wilcoxon $p$ in parentheses):

| Selection | BAGAN | BAGAN-GP | Δ recall | $F2$ $p$ |
|---|---|---|---|---|
| none | 0.839 | 0.896 | **+0.057** ($p$=6.1e-05) | $p$=0.0103 |
| margin | 0.824 | 0.849 | +0.025 ($p$=0.0302) | $p$=0.0413 |
| gate | 0.830 | 0.855 | +0.025 ($p$=0.0103) | $p$=0.0084 |

Positive at every level on both recall and $F2$, and every comparison clears $p$ < 0.05 — the only effect in this
study that does. This is also the largest effect size measured anywhere in the table.

### 3b. Per-class FID confirms the backbone effect, and explains why Proposed's FID
### does not differ from Variant 3's despite co-training updating the generator

The per-class breakdown (`ablation/kfold.py::_retrain_ab8c`) shows exactly why. Every mode that shares a backbone
and seed — `Baseline 1`, `Variant 1`, `Variant 2` on BAGAN; `Baseline 1a`, `Variant 1a`, `Variant 3`, `Proposed` on
BAGAN-GP — reports the identical c0/c1/c2/minority-mean, because none of them touch the generator after warm-up
except `Proposed`, and `Proposed`'s co-trained generator is discarded before it is ever measured.

**The generator genuinely is updated during co-training.** Each acquisition cycle takes `k_G` gradient steps on
it. What happens next is a built-in floor, not a missing feature: at the end of retraining the code computes a
real-anchored distance for both the co-trained generator and a frozen warm-up snapshot (`G_ref`), keeps whichever
is closer to real data, and logs the verdict as `[G-guard] ... (Ab-8b floor)`. The co-trained weights exist only
in memory for that one comparison — they are never written to disk, so the true co-trained FID cannot be
recovered without a code change and a re-run (out of scope for this notebook; see the per-seed G-guard line for
the comparison it made). **On all fifteen seeds in the study the guard chose the warm-up generator** — the
co-trained generator's real-anchored distance was 1.16–4.57× worse (mean
3.07×), narrowest at seed 45 and widest at seed 55 — which is why the FID columns
for `Proposed` and `Variant 3` are identical to three decimals in every row above: **this is the as-exported
FID, and it is what the pipeline actually produces**, not an artifact of how this table was built.

Two things follow from the aggregate:

**BAGAN → BAGAN-GP lowers minority FID from 180.28 ± 31.11 to
142.31 ± 8.68**, a ~21% improvement with well under half the
variance, and the same pattern holds class by class (c0: 189.6→96.0, c1:
168.3→128.5, c2: 192.2→156.1). This is the same backbone
effect already visible in recall and $F2$, now confirmed on an independent, downstream-blind metric — FID never
sees the classifier.

**Every BAGAN-GP row reports the identical FID, including `Proposed`.** That is the aggregate confirmation of the
single-seed G-guard finding, holding across all fifteen seeds: the floor always binds in this study, so
co-training's only effect on what the classifier sees (Insight 4) is through which *subset* of
warm-up-generator samples the acquisition loop selects, never through a different generator.

### 4. Co-training is not supported — and at n=15 this is a real null result, not a floor artifact

Proposed − Variant 3, per seed:

| seed | Δ recall | Δ $F2$ |
|---|---|---|
| 42 | +0.000 | +0.001 |
| 43 | -0.055 | -0.027 |
| 44 | +0.020 | +0.016 |
| 45 | -0.039 | -0.030 |
| 46 | +0.083 | +0.061 |
| 47 | -0.038 | -0.025 |
| 48 | +0.014 | -0.010 |
| 49 | -0.008 | -0.010 |
| 50 | +0.021 | +0.019 |
| 51 | -0.010 | -0.017 |
| 52 | +0.015 | -0.005 |
| 53 | +0.017 | +0.007 |
| 54 | -0.053 | -0.041 |
| 55 | +0.025 | -0.003 |
| 56 | -0.000 | -0.001 |
| **mean** | **-0.0005** | **-0.0043** |

The per-seed swings are an order of magnitude larger than the mean, and they change sign roughly as often as
not. A Wilcoxon signed-rank test confirms it directly: recall $p$ = 0.978, $F2$ $p$ = 0.252 — neither
comes close to significance, and unlike the n=5 study this is not because the test lacked power to detect
anything (n=15's floor is $p$ ≈ 6 × 10⁻⁵, and the backbone/precision effects above clear it easily). This is
noise, not a suppressed effect.

Independently, the **G-guard fired on all fifteen of the fifteen seeds**. The co-trained generator was
1.16–4.57× worse than the warm-up generator on the real-anchored distance every
time, so the pipeline discarded it. Co-training is not merely neutral — it degrades the generator, and the
method only holds up because a safeguard reverts it.

**A further check**: does the full `Proposed` pipeline (selection + co-training) even beat the backbone alone
with no selection? Proposed vs Baseline 1a on $F2$: $p$ = 0.330 — not significant, despite Proposed's
precision being significantly higher ($p$ = 6.1e-05). Selection alone (Variant 3) does reach
significance against Baseline 1a on $F2$ ($p$ = 0.022, Insight 2) — so co-training's presence in
`Proposed` appears to erode, not add to, the gain selection provides on its own.

### 5. Margin vs. the confidence gate: a tie whose sign flips, which is itself the evidence

Variant 1a (margin) reaches $F2$ 0.8398 ± 0.014; Variant 3 (gate) reaches
0.8414 ± 0.016. At five seeds margin led by 0.006; at fifteen, gate leads by
0.0016 — a sign flip on a gap smaller than either estimate's own noise. Neither
direction is statistically supported: $p$ = 0.489 on BAGAN-GP, $p$ = 0.978 on plain BAGAN
(Variant 1 vs Variant 2). A real, reproducible advantage for the confidence gate — the paper's headline
selection mechanism — would not change which rule wins depending on which ten seeds happen to be added.

### 6. Statistical testing: the n=5 floor is gone, and the results now split cleanly

At five paired seeds, a two-sided Wilcoxon signed-rank test bottomed out at $p$ = 0.0625 — no comparison could
reach $\alpha$ = 0.05 regardless of effect size, so every finding in the original notebook was necessarily
descriptive. At fifteen seeds the floor drops to roughly $p$ ≈ 6 × 10⁻⁵ (when all 15 seeds agree in sign), and
real hypothesis tests are possible for the first time in this study:

| Comparison | $p$ (Wilcoxon, n=15) | Verdict |
|---|---|---|
| Backbone effect (recall, all 3 selection levels) | 6.1e-05 / 0.030 / 0.010 | **Significant at every level** |
| Backbone effect ($F2$, all 3 selection levels) | 0.010 / 0.041 / 0.008 | **Significant at every level** |
| Selection precision gain (margin / gate vs none) | 6.1e-05 / 6.1e-05 | **Significant, both rules** |
| Selection $F2$ gain (margin / gate vs none) | 0.107 / 0.022 | Significant for gate only |
| Confidence gate vs margin ($F2$, either backbone) | 0.489 / 0.978 | Not significant |
| Co-training (Proposed vs Variant 3, recall / $F2$) | 0.978 / 0.252 | Not significant |
| Full pipeline vs backbone alone ($F2$) | 0.330 | Not significant |

No correction for multiple comparisons has been applied across the roughly ten tests run in this notebook;
treat the borderline results ($p$ between 0.02 and 0.04) as suggestive and the backbone/precision results
($p$ < 0.001) as solid. None of the non-significant results above are floor-limited — unlike the 5-seed study,
this notebook now has the power to detect effects of the size actually observed elsewhere in the table, and for
co-training, the gate-vs-margin question, and the full-pipeline-vs-backbone question, it does not find one.

---

## Summary

| Component | Verdict | Evidence |
|---|---|---|
| Gradient-penalised backbone | **Supported, significant at every level** | recall +0.025 to +0.057, $F2$ $p$ < 0.05 at every selection level (n=15); FID 180.3→142.3 (~21% lower) |
| Selection — precision & stability | **Supported, significant** | precision +0.123 to +0.136 over no-selection, both rules $p$ = 6.1e-05; $F2$ sd cut up to 2.2× |
| Selection — $F2$ gain over no-selection | **Mixed** | significant for confidence-gate ($p$ = 0.022), not for margin ($p$ = 0.107) |
| Confidence gate over margin | **Not supported** | statistically tied on both backbones ($p$ = 0.49, 0.98); nominal winner flips sign between the 5-seed and 15-seed batches |
| Three-player co-training | **Not supported — real null result at n=15** | recall/$F2$ change ~0, $p$ = 0.98 / 0.25; G-guard fires on 15/15 seeds, discarding the co-trained generator every time (1.2–4.6× worse than warm-up) |
| Full pipeline vs. backbone alone | **Not distinguishable on $F2$** | Proposed vs Baseline 1a: $p$ = 0.33, despite a significant precision gain ($p$ = 6.1e-05) |